In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
#pip install langchain-text-splitters langchain-community

Load & Split Your Documents

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

Embed & Store Vectors

In [ ]:
# Index chunks
_ = vector_store.add_documents(documents=all_splits)

In [ ]:
retriever = vector_store.as_retriever(
    search_type = "similarity",     # "mmr" or "similarity_score_threshold" also work
    search_kwargs = {"k": 4}
)

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate

SYSTEM = """You are an expert assistant.
Answer *only* from the context between <context></context>;
if the answer isn’t there, say “I don't know.”"""
USER = """<context>\n{context}\n</context>\n\nQuestion: {input}"""

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("user", USER)])

# NEW: wrap llm + prompt in a "stuff-documents" chain → Runnable
combine_docs_chain = create_stuff_documents_chain(llm, prompt)  

# Build the final RAG runnable
rag_chain = create_retrieval_chain(retriever, combine_docs_chain) 

In [ ]:
question = "What is Task Decomposition?"
result   = rag_chain.invoke({"input": question})

print(result["answer"])      # grounded answer
print(result["context"])     # the stuffed context string (if you need sources, see below)